In [2]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import datetime
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
from collections import defaultdict
import random
import re

In [3]:
main_df = pd.read_csv('./midterm/csis.csv',encoding='cp949')

In [4]:
main_df.head()

,Date of report,Target & Attacker,details,Target,group_name,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,26-Mar,European Commission (Attacker: Unknown),The European Commission claims it was targeted...,European Commission,Unknown,NaN,NaN,NaN,NaN,NaN
1,26-Mar,"Foster City, California (Attacker: Unknown)","Foster City, California was forced to pause al...","Foster City, California",Unknown,NaN,NaN,NaN,NaN,NaN
2,26-Mar,Stryker (Attacker: Handala),Medical device maker Stryker claimed a cyberat...,Stryker,Handala,NaN,NaN,NaN,NaN,NaN
3,26-Mar,Telus (Attacker: ShinyHunters),Canadian telecommunications firm Telus reporte...,Telus,ShinyHunters,NaN,NaN,NaN,NaN,NaN
4,26-Feb,Singapore Telecom Providers (Attacker: UNC3886),Singapore's Cyber Security Agency revealed tha...,Singapore Telecom Providers,UNC3886,NaN,NaN,NaN,NaN,NaN


In [5]:
main2_df = pd.read_csv('./midterm/group_technique_d.csv')

In [6]:
main2_df.head()

,group_stix_id,technique_stix_id,relationship_description,group_id,group_name,aliases,technique_id,technique_name,is_subtechnique,tactics,Date of report
0,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,attack-pattern--65f2d882-3f41-4d48-8a06-29af77...,[Indrik Spider](https://attack.mitre.org/group...,G0119,Indrik Spider,"['Indrik Spider', 'Evil Corp', 'Manatee Tempes...",T1003.001,LSASS Memory,True,['credential-access'],Jun-20
1,intrusion-set--b7f627e2-0817-4cd5-8d50-e75f8aa...,attack-pattern--10ffac09-e42d-4f56-ab20-db94c6...,[LuminousMoth](https://attack.mitre.org/groups...,G1014,LuminousMoth,['LuminousMoth'],T1539,Steal Web Session Cookie,False,['credential-access'],Jul-21
2,intrusion-set--918da025-04bd-48af-b6c4-f3e4d1b...,attack-pattern--f5d8eed6-48a9-4cdf-a3d7-d1ffa9...,[Medusa Group](https://attack.mitre.org/groups...,G1051,Medusa Group,['Medusa Group'],T1490,Inhibit System Recovery,False,['impact'],Jan-24
3,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,attack-pattern--635cbe30-392d-4e27-978e-667743...,[Wizard Spider](https://attack.mitre.org/group...,G0102,Wizard Spider,"['Wizard Spider', 'UNC1878', 'TEMP.MixMaster',...",T1136.001,Local Account,True,['persistence'],Oct-21
4,intrusion-set--3753cc21-2dae-4dfb-8481-d004e74...,attack-pattern--ef67e13e-5598-4adc-bdb2-998225...,[FIN7](https://attack.mitre.org/groups/G0046) ...,G0046,FIN7,"['FIN7', 'GOLD NIAGARA', 'ITG14', 'Carbon Spid...",T1204.001,Malicious Link,True,['execution'],Aug-21


In [7]:
group2_df = main2_df.groupby('group_name')

In [8]:
group2_df['technique_id'].count()

group_name
APT-C-36         38
APT1             23
APT12             5
APT16             1
APT17             2
                 ..
Winter Vivern    27
Wizard Spider    64
ZIRCONIUM        29
admin@338        12
menuPass         46
Name: technique_id, Length: 170, dtype: int64

In [9]:
d_main_df = main_df['Date of report']

In [10]:
d_main2_df = main2_df['Date of report']

In [11]:
combined_df = pd.concat([d_main_df, d_main2_df], ignore_index=True)

In [12]:
duplicate_count = combined_df.duplicated().sum()
print(f"총 중복 데이터 개수: {duplicate_count}개")

총 중복 데이터 개수: 5406개


In [13]:
duplicate_rows = combined_df[combined_df.duplicated(keep=False)]

# 확인하기
len(duplicate_rows)

5716

In [14]:
final_unique_df = combined_df.drop_duplicates(keep='first', ignore_index=True)

print(f"중복 제거 후 최종 데이터 개수: {len(final_unique_df)}개")

중복 제거 후 최종 데이터 개수: 349개


In [15]:
len(main_df)

1209

In [16]:
len(main2_df)

4546

In [17]:
main_df = pd.read_csv('./midterm/group_technique_d.csv')

In [18]:
main_df.head()

,group_stix_id,technique_stix_id,relationship_description,group_id,group_name,aliases,technique_id,technique_name,is_subtechnique,tactics,Date of report
0,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,attack-pattern--65f2d882-3f41-4d48-8a06-29af77...,[Indrik Spider](https://attack.mitre.org/group...,G0119,Indrik Spider,"['Indrik Spider', 'Evil Corp', 'Manatee Tempes...",T1003.001,LSASS Memory,True,['credential-access'],Jun-20
1,intrusion-set--b7f627e2-0817-4cd5-8d50-e75f8aa...,attack-pattern--10ffac09-e42d-4f56-ab20-db94c6...,[LuminousMoth](https://attack.mitre.org/groups...,G1014,LuminousMoth,['LuminousMoth'],T1539,Steal Web Session Cookie,False,['credential-access'],Jul-21
2,intrusion-set--918da025-04bd-48af-b6c4-f3e4d1b...,attack-pattern--f5d8eed6-48a9-4cdf-a3d7-d1ffa9...,[Medusa Group](https://attack.mitre.org/groups...,G1051,Medusa Group,['Medusa Group'],T1490,Inhibit System Recovery,False,['impact'],Jan-24
3,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,attack-pattern--635cbe30-392d-4e27-978e-667743...,[Wizard Spider](https://attack.mitre.org/group...,G0102,Wizard Spider,"['Wizard Spider', 'UNC1878', 'TEMP.MixMaster',...",T1136.001,Local Account,True,['persistence'],Oct-21
4,intrusion-set--3753cc21-2dae-4dfb-8481-d004e74...,attack-pattern--ef67e13e-5598-4adc-bdb2-998225...,[FIN7](https://attack.mitre.org/groups/G0046) ...,G0046,FIN7,"['FIN7', 'GOLD NIAGARA', 'ITG14', 'Carbon Spid...",T1204.001,Malicious Link,True,['execution'],Aug-21


In [19]:
len(main_df)

4546

In [20]:
group_df = main_df.groupby(['group_name','technique_id'])

In [21]:
group_df['technique_id'].describe()

count unique        top freq
group_name technique_id                             
APT-C-36   T1027            1      1      T1027    1
           T1027.003        1      1  T1027.003    1
           T1027.013        1      1  T1027.013    1
           T1027.016        1      1  T1027.016    1
           T1036.004        1      1  T1036.004    1
...                       ...    ...        ...  ...
menuPass   T1566.001        1      1  T1566.001    1
           T1568.001        1      1  T1568.001    1
           T1574.001        1      1  T1574.001    1
           T1583.001        1      1  T1583.001    1
           T1588.002        1      1  T1588.002    1

[4546 rows x 4 columns]

In [22]:
# group_name과 technique_id별 빈도수 계산 후 데이터프레임으로 변환
attack_counts = main_df.groupby(['group_name', 'technique_id']).size().reset_index(name='count')
attack_counts

,group_name,technique_id,count
0,APT-C-36,T1027,1
1,APT-C-36,T1027.003,1
2,APT-C-36,T1027.013,1
3,APT-C-36,T1027.016,1
4,APT-C-36,T1036.004,1
...,...,...,...
4541,menuPass,T1566.001,1
4542,menuPass,T1568.001,1
4543,menuPass,T1574.001,1
4544,menuPass,T1583.001,1


In [23]:
# group_name과 technique_id별 빈도수 계산 후 데이터프레임으로 변환
attack_counts = main_df.groupby(['technique_id','group_name']).size().reset_index(name='count')
attack_counts

,technique_id,group_name,count
0,T1001,Gamaredon Group,1
1,T1001.001,APT28,1
2,T1001.002,Axiom,1
3,T1001.003,Higaisa,1
4,T1001.003,Lazarus Group,1
...,...,...,...
4541,T1686.003,VOID MANTICORE,1
4542,T1690,APT38,1
4543,T1690,Medusa Group,1
4544,T1690,Sea Turtle,1


In [24]:
csia_df = pd.read_csv('./midterm/csia_origin.csv',encoding='utf-8')

In [25]:
csia_df.head(3)

,국가,공격코드(경보),출판일,제목,설명,상세URL,상세본문
0,중국,AA26-113A,"April 23, 2026",Defending Against China-Nexus Covert Networks ...,"The advisory outlines tactics, techniques, pro...",https://www.cisa.gov/news-events/cybersecurity...,Defending against china-nexus covert networks ...
1,중국,AA25-239A,"August 27, 2025",Countering Chinese State-Sponsored Actors Comp...,"The National Security Agency (NSA), Cybersecur...",https://www.cisa.gov/news-events/cybersecurity...,Executive summary\nPeople’s Republic of China ...
2,중국,NaN,"December 18, 2024",Mobile Communications Best Practice Guidance,This guidance was crafted in response to ident...,https://www.cisa.gov/resources-tools/resources...,This guidance was crafted in response to ident...


In [26]:
t_m_df = pd.read_csv('./midterm/techniques.csv')

In [27]:
t_m_df.head()

,technique_stix_id,technique_id,technique_name,is_subtechnique,tactics,description
0,attack-pattern--0042a9f5-f053-4769-b3ef-9ad018...,T1055.011,Extra Window Memory Injection,True,"['stealth', 'privilege-escalation']",Adversaries may inject malicious code into pro...
1,attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...,T1053.005,Scheduled Task,True,"['execution', 'persistence', 'privilege-escala...",Adversaries may abuse the Windows Task Schedul...
2,attack-pattern--005cc321-08ce-4d17-b1ea-cb5275...,T1205.002,Socket Filters,True,"['stealth', 'persistence', 'command-and-control']",Adversaries may attach filters to a network so...
3,attack-pattern--00f90846-cbd1-4fc5-9233-df5c2b...,T1560.001,Archive via Utility,True,['collection'],Adversaries may use utilities to compress and/...
4,attack-pattern--01327cde-66c4-4123-bf34-5f258d...,T1021.005,VNC,True,['lateral-movement'],Adversaries may use [Valid Accounts](https://a...


In [28]:
t_df = t_m_df[~t_m_df['technique_id'].str.contains(r'\.', na=False)]

In [29]:
t_df.head()

,technique_stix_id,technique_id,technique_name,is_subtechnique,tactics,description
5,attack-pattern--01a5a209-b94c-450b-b7f9-946497...,T1047,Windows Management Instrumentation,False,['execution'],Adversaries may abuse Windows Management Instr...
6,attack-pattern--01c9b54f-c04e-41ba-b0c3-cfe784...,T1687,Exploitation for Defense Impairment,False,['defense-impairment'],Adversaries may exploit vulnerabilities in sec...
7,attack-pattern--0259baeb-9f63-4c69-bf10-eb038c...,T1113,Screen Capture,False,['collection'],Adversaries may attempt to take screen capture...
9,attack-pattern--03259939-0b57-482f-8eb5-87c0e0...,T1037,Boot or Logon Initialization Scripts,False,"['persistence', 'privilege-escalation']",Adversaries may use scripts automatically exec...
10,attack-pattern--035bb001-ab69-4a0b-9f6c-2de8b0...,T1557,Adversary-in-the-Middle,False,"['credential-access', 'collection']",Adversaries may attempt to position themselves...


사보타주 : impact
독싱 : reconnaissance, collection, exfiltration
금전적 탈취 : collection, exfiltration, impact 

In [30]:
t_df['tactics'].value_counts()

tactics
['discovery']                                                           31
['stealth']                                                             20
['command-and-control']                                                 16
['collection']                                                          15
['defense-impairment']                                                  15
['execution']                                                           15
['impact']                                                              15
['credential-access']                                                   13
['reconnaissance']                                                      12
['resource-development']                                                 9
['persistence']                                                          9
['exfiltration']                                                         9
['initial-access']                                                       7
['lateral-movemen

In [31]:
# 조건 1: 무조건 'impact'를 포함해야 함
cond_impact = t_df['tactics'].str.contains('impact', case=False, na=False)

# 조건 2: 나머지 세 개(defense-impairment, execution, stealth) 중 최소 하나를 포함해야 함
cond_others = t_df['tactics'].str.contains('defense-impairment|execution|stealth', case=False, na=False)

# 두 조건을 동시에 만족(&)하는 행만 추출
sabotage_candidates = t_df[cond_impact & cond_others]

In [32]:
sabotage_candidates.head()

,technique_stix_id,technique_id,technique_name,is_subtechnique,tactics,description


In [33]:
cond_impact = t_df['tactics'].str.contains('impact', case=False, na=False)
cond_impact.head()

5     False
6     False
7     False
9     False
10    False
Name: tactics, dtype: bool

In [34]:
# 4가지 전술 중 하나라도 포함하는 행을 한 번에 추출 (OR 조건)
tactics_combined = 'impact|defense-impairment|execution|stealth'

sabotage_candidates = t_df[t_df['tactics'].str.contains(tactics_combined, case=False, na=False)]
sabotage_candidates.count()

technique_stix_id    80
technique_id         80
technique_name       80
is_subtechnique      80
tactics              80
description          80
dtype: int64

In [35]:
#impact가 주요한듯
tactics_combined = 'impact'

sabotage_candidates = t_df[t_df['tactics'].str.contains(tactics_combined, case=False, na=False)]
sabotage_candidates.count()

technique_stix_id    15
technique_id         15
technique_name       15
is_subtechnique      15
tactics              15
description          15
dtype: int64

In [36]:
tactics_combined= 'reconnaissance|collection|exfiltration'
doxing_candidates = t_df[t_df['tactics'].str.contains(tactics_combined, case=False, na=False)]
doxing_candidates.count()


technique_stix_id    38
technique_id         38
technique_name       38
is_subtechnique      38
tactics              38
description          38
dtype: int64

In [37]:
tactics_combined= 'collection|exfiltration|impact'
finance_candidates = t_df[t_df['tactics'].str.contains(tactics_combined, case=False, na=False)]
finance_candidates.count()


technique_stix_id    41
technique_id         41
technique_name       41
is_subtechnique      41
tactics              41
description          41
dtype: int64

In [38]:
finance_candidates['description'].head()

7     Adversaries may attempt to take screen capture...
10    Adversaries may attempt to position themselves...
30    Adversaries may collect data related to manage...
49    An adversary can leverage a computer's periphe...
68    Adversaries may target user email to collect s...
Name: description, dtype: object

In [39]:
must_have_keywords = ['wipe', 'destroy', 'render unusable', 'corrupt', 'brick', 'erase', 'disrupt']
weight_keywords = ['rm -rf', 'shred', 'format', 'vssadmin', 'shadowcopy', 'mbr', 'overwrite', 'stop', 'kill']
negative_keywords = ['ransom', 'decrypt', 'extortion', 'payment', 'bitcoin', 'monetary']

def score_sabotage(description):
    if pd.isna(description):
        return 0
    
    desc_lower = str(description).lower()
    
    # 규칙 1: 랜섬웨어/금융 갈취 단어가 있으면 사보타주에서 제외 (0점)
    if any(neg in desc_lower for neg in negative_keywords):
        return 0
        
    # 규칙 2: 1순위(필수 파괴 동사)가 최소 하나는 들어있어야 함 (없으면 0점)
    if not any(must in desc_lower for must in must_have_keywords):
        return 0
    
    # 규칙 3: 조건을 만족했다면 점수 계산 (1순위 개수 * 5점 + 2순위 개수 * 2점)
    score = 0
    score += sum(5 for must in must_have_keywords if must in desc_lower)
    score += sum(2 for weight in weight_keywords if weight in desc_lower)
    
    return score

# [실행] 기존에 구한 대분류 + Tactic 필터링된 데이터프레임에 적용
sabotage_candidates['score'] = sabotage_candidates['description'].apply(score_sabotage)

# 점수가 0점보다 큰 것(사보타주로 판명된 것)만 남기고 점수 높은 순으로 정렬
final_sabotage = sabotage_candidates[sabotage_candidates['score'] > 0].sort_values(by='score', ascending=False)

# 결과 출력
print(final_sabotage[['technique_id','score']])

    technique_id  score
80         T1561     19
693        T1529     15
581        T1485     12
100        T1489      7
668        T1495      7


C:\Users\smart\AppData\Local\Temp\ipykernel_28084\2401076672.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sabotage_candidates['score'] = sabotage_candidates['description'].apply(score_sabotage)


In [40]:
# technique_id가 'T1561'인 행의 'description' 값 가져오기
t1561_desc = t_df[t_df['technique_id'] == 'T1561']['description'].values[0]

print(t1561_desc)

Adversaries may wipe or corrupt raw disk data on specific systems or in large numbers in a network to interrupt availability to system and network resources. With direct write access to a disk, adversaries may attempt to overwrite portions of disk data. Adversaries may opt to wipe arbitrary portions of disk data and/or wipe disk structures like the master boot record (MBR). A complete wipe of all disk sectors may be attempted.

To maximize impact on the target organization in operations where network-wide availability interruption is the goal, malware used for wiping disks may have worm-like features to propagate across a network by leveraging additional techniques like [Valid Accounts](https://attack.mitre.org/techniques/T1078), [OS Credential Dumping](https://attack.mitre.org/techniques/T1003), and [SMB/Windows Admin Shares](https://attack.mitre.org/techniques/T1021/002).(Citation: Novetta Blockbuster Destructive Malware)

On network devices, adversaries may wipe configuration files 

In [41]:
must_have_keywords = ['dox', 'doxx', 'leak', 'expose', 'dump', 'publish', 'release', 'publicly share', 'disclose']

# 가중치 키워드: 유출되는 데이터의 성격이나 목적 (개인정보, 평판 훼손 등)
weight_keywords = ['pii', 'personal information', 'sensitive', 'private', 'identity', 'credentials', 'embarrass', 'reputation', 'pastebin', 'doxbin', 'shame']

# 제외 키워드: 단순 파괴(사보타주)나 단순 암호화(랜섬웨어) 목적인 경우 감점/제외
negative_keywords = ['encrypt', 'wiper', 'wipe', 'corrupt', 'brick', 'locker']

def score_sabotage(description):
    if pd.isna(description):
        return 0
    
    desc_lower = str(description).lower()
    
    # 규칙 1: 랜섬웨어/금융 갈취 단어가 있으면 사보타주에서 제외 (0점)
    if any(neg in desc_lower for neg in negative_keywords):
        return 0
        
    # 규칙 2: 1순위(필수 파괴 동사)가 최소 하나는 들어있어야 함 (없으면 0점)
    if not any(must in desc_lower for must in must_have_keywords):
        return 0
    
    # 규칙 3: 조건을 만족했다면 점수 계산 (1순위 개수 * 5점 + 2순위 개수 * 2점)
    score = 0
    score += sum(5 for must in must_have_keywords if must in desc_lower)
    score += sum(2 for weight in weight_keywords if weight in desc_lower)
    
    return score

# [실행] 기존에 구한 대분류 + Tactic 필터링된 데이터프레임에 적용
doxing_candidates['score'] = doxing_candidates['description'].apply(score_sabotage)

# 점수가 0점보다 큰 것(사보타주로 판명된 것)만 남기고 점수 높은 순으로 정렬
final_doxing = doxing_candidates[doxing_candidates['score'] > 0].sort_values(by='score', ascending=False)

# 결과 출력
print(final_doxing[['technique_id','score']])

    technique_id  score
151        T1530     18
573        T1213     16
245        T1589     16
426        T1590     10
413        T1591     10
30         T1602      7
72         T1594      7
20         T1592      5
254        T1596      5


C:\Users\smart\AppData\Local\Temp\ipykernel_28084\1851189993.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  doxing_candidates['score'] = doxing_candidates['description'].apply(score_sabotage)


In [42]:
import pandas as pd

# 1. 금전적 갈취 식별을 위한 키워드 세트
# 필수 키워드: 금전적 목적, 랜섬웨어, 협박, 지불 등 직접적인 금전 동기를 나타내는 단어
must_have_keywords = ['ransom', 'extort', 'extortion', 'financial', 'monetary', 'payment', 'monetize', 'lucrative', 'pay']

# 가중치 키워드: 타겟이 되는 금융 시스템이나 결제 수단 (암호화폐, 은행, 지갑 등)
weight_keywords = ['cryptocurrency', 'crypto', 'bitcoin', 'btc', 'monero', 'wallet', 'bank', 'swift', 'atm', 'money', 'funds', 'transfer', 'theft', 'steal', 'heist']

# 제외 키워드: 순수 국가 간첩 활동이나 정치적 시위 목적이면 감점/제외
negative_keywords = ['espionage', 'hacktivism', 'political', 'protest', 'state secret', 'dissident']

def score_finance(description):
    if pd.isna(description):
        return 0
    
    desc_lower = str(description).lower()
    
    # 규칙 1: 순수 간첩 활동이나 정치적 시위 단어가 주를 이루면 제외 (0점)
    if any(neg in desc_lower for neg in negative_keywords):
        return 0    
    # 규칙 2: 1순위(필수 금전 동기 단어)가 최소 하나는 들어있어야 함 (없으면 0점)
    if not any(must in desc_lower for must in must_have_keywords):
        return 0
    # 규칙 3: 조건을 만족했다면 점수 계산 (1순위 개수 * 5점 + 2순위 개수 * 2점)
    score = 0
    score += sum(5 for must in must_have_keywords if must in desc_lower)
    score += sum(2 for weight in weight_keywords if weight in desc_lower)
    
    return score

# [실행] 필터링해두신 finance_candidates 데이터프레임에 바로 적용
# pandas 경고(SettingWithCopyWarning)를 방지하기 위해 .copy()를 사용해 독립적인 데이터프레임으로 만듭니다.
finance_candidates = finance_candidates.copy()

finance_candidates['score'] = finance_candidates['description'].apply(score_finance)

# 점수가 0점보다 큰 것(금전적 목적으로 판명된 것)만 남기고 높은 순으로 정렬
final_finance = finance_candidates[finance_candidates['score'] > 0].sort_values(by='score', ascending=False)

# 결과 출력
print(final_finance[['technique_id', 'score']].head(10))

    technique_id  score
381        T1657     53
513        T1667     14
151        T1530     10
490        T1486     10
474        T1531      5
669        T1490      5


'Denial of service' : 'T1498',
    'Espionage' : 'T1020',
    'Defacement' : 'T1491',
    'Data destruction' : 'T1485',
    'Sabotage' : 'T1561',
    'Doxing' :'T1530',
    'Financial Theft' : 'T1657' 

In [ ]:
mainc_df = pd.read_csv('./midterm/cot_china.csv')

In [ ]:
mainc_df['Type of incident'].value_counts()

Type of incident
Espionage            250
Sabotage               2
Denial of service      2
Financial Theft        1
Data destruction       1
Name: count, dtype: int64

In [ ]:
mainr_df = pd.read_csv('./midterm/cot_russia.csv')

In [ ]:
mainr_df['Type of incident'].value_counts()

Type of incident
Espionage                      139
Sabotage                        16
Data destruction                15
Denial of service                8
Doxing                           5
Defacement                       4
 and civil society targets"      1
Name: count, dtype: int64

In [ ]:
mainr_df = mainr_df[mainr_df['Type of incident'] != 'Government']

In [ ]:
mainr_df.to_csv('cot_russia.csv')

In [ ]:
maini_df = pd.read_csv('./midterm/cot_iran.csv',encoding='cp949')
maini_df['Type of incident'].value_counts()

Type of incident
Espionage            87
Data destruction      7
Financial Theft       6
Sabotage              3
Denial of service     2
Name: count, dtype: int64

In [ ]:
mainn_df = pd.read_csv('./midterm/cot_northkorea.csv')
mainn_df['Type of incident'].value_counts()

Type of incident
Espionage            72
Financial Theft      25
Doxing                2
Denial of service     2
Data destruction      1
Name: count, dtype: int64

In [ ]:
incident_mapping = {
    'Denial of service': 'T1498',
    'Espionage': 'T1020',
    'Defacement': 'T1491',
    'Data destruction': 'T1485',
    'Sabotage': 'T1561',
    'Doxing': 'T1530',
    'Financial Theft': 'T1657'
}

# 2. map() 함수를 사용해 'technique_id'라는 새로운 컬럼을 만듭니다.
mainc_df['technique_id'] = mainc_df['Type of incident'].map(incident_mapping)

# 3. (선택) 딕셔너리에 없는 값이 있어서 빈칸(NaN)이 된 경우 처리하기
# 만약 매핑 리스트에 없는 희귀한 유형이 섞여 있다면 빈칸으로 남게 됩니다.
# 빈칸을 'Unknown'이나 'Mapping Failed'로 채우고 싶다면 아래 코드를 주석 해제하세요.
# main_df['technique_id'] = main_df['technique_id'].fillna('Unknown')

# 결과 확인하기 (두 컬럼이 잘 짝지어졌는지 눈으로 확인)
print(mainc_df[['Type of incident', 'technique_id']].head(10))

  Type of incident technique_id
0        Espionage        T1020
1        Espionage        T1020
2        Espionage        T1020
3        Espionage        T1020
4        Espionage        T1020
5        Espionage        T1020
6        Espionage        T1020
7        Espionage        T1020
8        Espionage        T1020
9        Espionage        T1020


In [ ]:
incident_mapping = {
    'Denial of service': 'T1498',
    'Espionage': 'T1020',
    'Defacement': 'T1491',
    'Data destruction': 'T1485',
    'Sabotage': 'T1561',
    'Doxing': 'T1530',
    'Financial Theft': 'T1657'
}

# 2. map() 함수를 사용해 'technique_id'라는 새로운 컬럼을 만듭니다.
mainr_df['technique_id'] = mainr_df['Type of incident'].map(incident_mapping)

# 3. (선택) 딕셔너리에 없는 값이 있어서 빈칸(NaN)이 된 경우 처리하기
# 만약 매핑 리스트에 없는 희귀한 유형이 섞여 있다면 빈칸으로 남게 됩니다.
# 빈칸을 'Unknown'이나 'Mapping Failed'로 채우고 싶다면 아래 코드를 주석 해제하세요.
# main_df['technique_id'] = main_df['technique_id'].fillna('Unknown')

# 결과 확인하기 (두 컬럼이 잘 짝지어졌는지 눈으로 확인)
print(mainr_df[['Type of incident', 'technique_id']].head(10))

    Type of incident technique_id
0          Espionage        T1020
1  Denial of service        T1498
2          Espionage        T1020
3          Espionage        T1020
4          Espionage        T1020
5          Espionage        T1020
6           Sabotage        T1561
7   Data destruction        T1485
8          Espionage        T1020
9   Data destruction        T1485


In [ ]:
incident_mapping = {
    'Denial of service': 'T1498',
    'Espionage': 'T1020',
    'Defacement': 'T1491',
    'Data destruction': 'T1485',
    'Sabotage': 'T1561',
    'Doxing': 'T1530',
    'Financial Theft': 'T1657'
}

# 2. map() 함수를 사용해 'technique_id'라는 새로운 컬럼을 만듭니다.
maini_df['technique_id'] = maini_df['Type of incident'].map(incident_mapping)

# 3. (선택) 딕셔너리에 없는 값이 있어서 빈칸(NaN)이 된 경우 처리하기
# 만약 매핑 리스트에 없는 희귀한 유형이 섞여 있다면 빈칸으로 남게 됩니다.
# 빈칸을 'Unknown'이나 'Mapping Failed'로 채우고 싶다면 아래 코드를 주석 해제하세요.
# main_df['technique_id'] = main_df['technique_id'].fillna('Unknown')

# 결과 확인하기 (두 컬럼이 잘 짝지어졌는지 눈으로 확인)
print(maini_df[['Type of incident', 'technique_id']].head(10))

   Type of incident technique_id
0  Data destruction        T1485
1         Espionage        T1020
2         Espionage        T1020
3         Espionage        T1020
4         Espionage        T1020
5  Data destruction        T1485
6         Espionage        T1020
7         Espionage        T1020
8         Espionage        T1020
9         Espionage        T1020


In [ ]:
incident_mapping = {
    'Denial of service': 'T1498',
    'Espionage': 'T1020',
    'Defacement': 'T1491',
    'Data destruction': 'T1485',
    'Sabotage': 'T1561',
    'Doxing': 'T1530',
    'Financial Theft': 'T1657'
}

# 2. map() 함수를 사용해 'technique_id'라는 새로운 컬럼을 만듭니다.
mainn_df['technique_id'] = mainn_df['Type of incident'].map(incident_mapping)

# 3. (선택) 딕셔너리에 없는 값이 있어서 빈칸(NaN)이 된 경우 처리하기
# 만약 매핑 리스트에 없는 희귀한 유형이 섞여 있다면 빈칸으로 남게 됩니다.
# 빈칸을 'Unknown'이나 'Mapping Failed'로 채우고 싶다면 아래 코드를 주석 해제하세요.
# main_df['technique_id'] = main_df['technique_id'].fillna('Unknown')

# 결과 확인하기 (두 컬럼이 잘 짝지어졌는지 눈으로 확인)
print(mainn_df[['Type of incident', 'technique_id']].head(10))

  Type of incident technique_id
0        Espionage        T1020
1  Financial Theft        T1657
2        Espionage        T1020
3        Espionage        T1020
4        Espionage        T1020
5        Espionage        T1020
6        Espionage        T1020
7  Financial Theft        T1657
8  Financial Theft        T1657
9  Financial Theft        T1657


In [ ]:
mainr_df.to_csv('cot_russia.csv')
maini_df.to_csv('cot_iran.csv')
mainn_df.to_csv('cot_northkorea.csv')
mainc_df.to_csv('cot_china.csv')

In [ ]:
mainc_df

In [ ]:
import pandas as pd
import ast

# 데이터 불러오기 (경로는 환경에 맞게 수정하세요)
df = pd.read_csv('./midterm/group_technique_country.csv')

def get_broad_category(tactics_str):
    if pd.isna(tactics_str):
        return 'General / Dual-Use'
    
    # 텍스트 형태의 리스트 "['impact']"를 실제 파이썬 리스트로 변환 혹은 소문자 텍스트로 처리
    tactics_lower = str(tactics_str).lower()
    
    # 1. 파괴 계열 (Impact)
    if 'impact' in tactics_lower:
        return 'Destructive (Needs NLP for DoS, Defacement, Destruction, Sabotage)'
    
    # 2. 탈취 계열 (Collection, Exfiltration)
    elif 'exfiltration' in tactics_lower or 'collection' in tactics_lower:
        return 'Theft/Leak (Needs NLP for Espionage, Doxing, Financial)'
    
    # 3. 그 외 초기 침투 및 유지 기술
    else:
        return 'General / Dual-Use'

# 새로운 1차 분류 컬럼 생성
df['Broad_Category'] = df['tactics'].apply(get_broad_category)

# 결과 확인 (어떤 기술들이 파괴/탈취로 나뉘었는지 카운트 확인)
print(df['Broad_Category'].value_counts())

Broad_Category
General / Dual-Use                                                    4061
Theft/Leak (Needs NLP for Espionage, Doxing, Financial)                401
Destructive (Needs NLP for DoS, Defacement, Destruction, Sabotage)      84
Name: count, dtype: int64


In [ ]:
import pandas as pd
import re

# 1. 7개 그룹별 정밀 가중치 키워드 사전 (Dictionary) 구성
# high: 핵심 동사/명사 (+5점), low: 보조 문맥 단어 (+2점), neg: 오탐 방지용 감점 단어 (-5점)
weighted_rules = {
    'Financial Theft': {
        'high': ['ransom', 'extort', 'financial', 'monetary', 'payment', 'lucrative'],
        'low': ['crypto', 'bitcoin', 'wallet', 'bank', 'theft', 'steal money'],
        'neg': ['espionage', 'hacktivism', 'political']
    },
    'Espionage': {
        'high': ['exfiltrate', 'espionage', 'intel', 'spy', 'steal data', 'state secret'],
        'low': ['gather', 'monitor', 'reconnaissance', 'collection', 'surveillance', 'sensitive information'],
        'neg': ['ransom', 'destroy', 'wipe']
    },
    'Data destruction': {
        'high': ['wipe', 'destroy', 'unrecoverable', 'erase', 'corrupt'],
        'low': ['overwrite', 'delete', 'mbr', 'format', 'render unusable'],
        'neg': ['ransom', 'decrypt'] # 랜섬웨어가 데이터를 암호화하는 것을 파괴로 오해하지 않도록 방어
    },
    'Sabotage': {
        'high': ['sabotage', 'disrupt', 'brick', 'disable', 'halt', 'stop service'],
        'low': ['impair', 'degrade', 'availability', 'shut down'],
        'neg': ['ransom']
    },
    'Denial of service': {
        'high': ['dos', 'ddos', 'volumetric', 'network exhaustion'],
        'low': ['flood', 'traffic', 'botnet', 'overwhelm'],
        'neg': []
    },
    'Doxing': {
        'high': ['dox', 'doxx', 'leak', 'publicly share', 'expose'],
        'low': ['publish', 'dump', 'reputation', 'pastebin', 'shame', 'pii'],
        'neg': ['encrypt', 'wipe']
    },
    'Defacement': {
        'high': ['deface', 'graffiti', 'website alter'],
        'low': ['modify visual', 'visual representation', 'message'],
        'neg': []
    }
}

# 2. 텍스트를 추출하여 가중치 점수를 계산하는 메인 함수
def calculate_weighted_category(description):
    if pd.isna(description):
        return 'General / Dual-Use'
        
    # 특수문자 제거 및 소문자 변환 (정확한 매칭을 위한 전처리)
    desc_clean = re.sub(r'[^a-zA-Z0-9\s]', ' ', str(description)).lower()
    
    # 카테고리별 최종 점수를 담을 딕셔너리
    scores = {category: 0 for category in weighted_rules.keys()}
    
    for category, rules in weighted_rules.items():
        # High 가중치 (+5)
        for word in rules['high']:
            if word in desc_clean:
                scores[category] += (desc_clean.count(word) * 5)
                
        # Low 가중치 (+2)
        for word in rules['low']:
            if word in desc_clean:
                scores[category] += (desc_clean.count(word) * 2)
                
        # # Negative 감점 (-5)
        # for word in rules['neg']:
        #     if word in desc_clean:
        #         scores[category] -= (desc_clean.count(word) * 5)
    
    # 3. 점수 판별 로직
# 3. 점수 판별 로직 (여기서부터 수정!)
    max_score = max(scores.values())
    
    # 안전장치: 어떤 키워드도 안 걸렸거나, 감점 때문에 최고 점수가 0 이하가 된 경우
    if max_score <= 0:
        return 'Unknown'
        
    # 최고 점수를 받은 카테고리 추출
    best_categories = [cat for cat, score in scores.items() if score == max_score]
    
    # [핵심 변경] 듀얼(A & B) 방지: 동점이 몇 개든 상관없이 무조건 첫 번째 1개만 강제로 뽑아냄
    return best_categories[0]

# ==========================================
# (이 아래 실행 코드는 기존과 동일하게 사용하면 돼!)
# df['Final_Category'] = df['relationship_description'].apply(calculate_weighted_category)

# ==========================================
# [실행 및 테스트]
# ==========================================
# mitre_df는 분석할 MITRE 데이터프레임이라고 가정합니다.
# mitre_df['Final_Category'] = mitre_df['description'].apply(calculate_weighted_category)

# 샘플 데이터로 성능 검증
import pandas as pd

# 1. 실전 데이터 불러오기 (말씀하신 경로 그대로 적용)
df = pd.read_csv('./midterm/group_technique_country.csv')

# 2. 'relationship_description' 컬럼에 가중치 기반 자동 분류 함수 적용
# apply()가 description 안의 텍스트를 하나하나 읽어서 점수를 매기고 카테고리를 뱉어냅니다.
df['Final_Category'] = df['relationship_description'].apply(calculate_weighted_category)

# 3. 결과 확인하기 
# technique_id와 원본 설명, 그리고 모델이 판별한 최종 카테고리를 나란히 비교해 봅니다.
print("=== 자연어 가중치 분석 완료 ===")
print(df[['technique_id', 'relationship_description', 'Final_Category']].head(10))

# 4. (보너스) 각 카테고리별로 데이터가 몇 개씩 분류되었는지 통계 확인
print("\n=== 카테고리별 분류 통계 ===")
print(df['Final_Category'].value_counts())

=== 자연어 가중치 분석 완료 ===
  technique_id                           relationship_description  \
0        T1003  [Indrik Spider](https://attack.mitre.org/group...   
1        T1539  [LuminousMoth](https://attack.mitre.org/groups...   
2        T1490  [Medusa Group](https://attack.mitre.org/groups...   
3        T1136  [Wizard Spider](https://attack.mitre.org/group...   
4        T1204  [FIN7](https://attack.mitre.org/groups/G0046) ...   
5        T1027  [UNC3886](https://attack.mitre.org/groups/G104...   
6        T1059  [WIRTE](https://attack.mitre.org/groups/G0090)...   
7        T1684  [Star Blizzard](https://attack.mitre.org/group...   
8        T1571  [WIRTE](https://attack.mitre.org/groups/G0090)...   
9        T1681  [UNC3886](https://attack.mitre.org/groups/G104...   

    Final_Category  
0           Doxing  
1          Unknown  
2  Financial Theft  
3          Unknown  
4          Unknown  
5        Espionage  
6          Unknown  
7          Unknown  
8          Unknown  
9       

In [49]:
import pandas as pd
import re

# 1. 실전 데이터 불러오기
df = pd.read_csv('./midterm/group_technique_country.csv')

# 2. 7개 그룹별 정밀 가중치 키워드 사전 정의
weighted_rules = {
    'Financial Theft': {
        'high': [
            'ransom', 'extort', 'financial', 'monetary', 'payment', 'lucrative', 'pay', 'heist', 
            'ransomware', 'double extortion', 'extortionist', 'monetize', 'pos theft', 'credit card'
        ],
        'low': [
            'crypto', 'bitcoin', 'wallet', 'bank', 'theft', 'steal money', 'btc', 'monero', 'atm', 
            'swift', 'funds', 'transfer', 'cardholder', 'xmr', 'darkside', 'revil', 'lockbit', 
            'conti', 'clop', 'blackcat', 'cipher', 'demand', 'negotiation', 'fin7', 'fin11', 
            'cryptomining', 'miner', 'coinhive', 'xmrig', 'accounting', 'transaction'
        ],
        'neg': ['espionage', 'hacktivism', 'political', 'state secret']
    },
    'Espionage': {
        'high': [
            'exfiltrate', 'espionage', 'intel', 'spy', 'steal data', 'state secret', 'reconnaissance', 
            'gathering', 'intelligence', 'cyber espionage', 'apt', 'state sponsored', 'trade secret'
        ],
        'low': [
            'gather', 'monitor', 'collection', 'surveillance', 'sensitive', 'credentials', 'password', 
            'keylogger', 'lsass', 'mimikatz', 'cookie', 'token', 'cloud hopper', 'theft data', 'harvest', 
            'keylog', 'credential access', 'proc dump', 'procdump', 'dumping', 'sam database', 'ntds', 
            'laurel', 'cobalt strike', 'brute force', 'phishing', 'spearphishing', 'confidential', 
            'document steal', 'blueprint', 'geopolitical', 'recon', 'scanning', 'network map'
        ],
        'neg': ['ransom', 'destroy', 'wipe', 'deface']
    },
    'Data destruction': {
        'high': [
            'wipe', 'destroy', 'unrecoverable', 'erase', 'corrupt', 'destruction', 'wiper', 
            'data destruction', 'disk wipe', 'file deletion'
        ],
        'low': [
            'overwrite', 'delete', 'mbr', 'format', 'render unusable', 'shred', 'vssadmin', 
            'shadowcopy', 'purged', 'notforransom', 'hermeticwiper', 'caddywiper', 'whispergate', 
            'industroyer', 'zero out', 'nullify', 'raw disk', 'partition destroy', 'fsutil', 
            'rmdir', 'del tree', 'eraser'
        ],
        'neg': ['ransom', 'decrypt', 'payment', 'extort']
    },
    'Sabotage': {
        'high': [
            'sabotage', 'disrupt', 'brick', 'disable', 'halt', 'stop service', 'disruption', 
            'scada attack', 'industrial control', 'operational technology'
        ],
        'low': [
            'impair', 'degrade', 'availability', 'shut down', 'kill', 'terminate', 'uninstall', 
            'tamper', 'stuxnet', 'plc', 'ics', 'modbus', 'blackout', 'power grid', 'firmware flash', 
            'registry corrupt', 'process kill', 'service stop', 'net stop', 'taskkill', 'block access', 
            'interference', 'malfunction'
        ],
        'neg': ['ransom', 'payment']
    },
    'Denial of service': {
        'high': [
            'dos', 'ddos', 'volumetric', 'network exhaustion', 'denial of service', 
            'distributed denial of service', 'botnet'
        ],
        'low': [
            'flood', 'traffic', 'overwhelm', 'amplification', 'syn', 'udp', 'icmp', 'http flood', 
            'slowloris', 'dns amplification', 'ntp amplification', 'mirai', 'layer 7', 'layer 4', 
            'bandwidth', 'ping of death', 'spoofing traffic', 'clog', 'unresponsive'
        ],
        'neg': []
    },
    'Doxing': {
        'high': [
            'dox', 'doxx', 'leak', 'publicly share', 'expose', 'doxing', 'doxxing', 'public exposure'
        ],
        'low': [
            'publish', 'dump', 'reputation', 'pastebin', 'shame', 'pii', 'doxbin', 'identity', 
            'embarrass', 'blackmail', 'naming and shaming', 'leak site', 'onion site', 'telegram channel', 
            'personal record', 'home address', 'phone number', 'email leak', 'social media expose'
        ],
        'neg': ['encrypt', 'wipe', 'brick', 'dos']
    },
    'Defacement': {
        'high': [
            'deface', 'graffiti', 'website alter', 'defacement', 'hacktivist defacement'
        ],
        'low': [
            'modify visual', 'visual representation', 'message', 'altered', 'appearance', 'index html', 
            'homepage', 'visual modify', 'political banner', 'slogan', 'hacked by', 'hacktivism banner', 
            'replace image', 'web server modify'
        ],
        'neg': ['exfiltrate', 'spy', 'ransom']
    }
}

# 3. Tactics 힌트와 키워드 점수를 결합한 하이브리드 분류 함수
def calculate_hybrid_category(row):
    description = row['relationship_description']
    # tactics 컬럼 데이터 안전하게 소문자 텍스트로 변환
    tactics_str = str(row['tactics']).lower() if not pd.isna(row['tactics']) else ''
    
    # 설명문(description) 자체가 비어있으면 매칭 불가하므로 Unknown 처리
    if pd.isna(description):
        return 'Unknown'
        
    # 특수문자 제거 및 소문자 변환 (정확한 단어 매칭용 전처리)
    desc_clean = re.sub(r'[^a-zA-Z0-9\s]', ' ', str(description)).lower()
    
    # 카테고리별 점수판 초기화
    scores = {category: 0 for category in weighted_rules.keys()}
    
    # [1단계] 본문 텍스트 기반 가중치 점수 계산
    for category, rules in weighted_rules.items():
        # High 가중치 (+5점)
        for word in rules['high']:
            if word in desc_clean:
                scores[category] += (desc_clean.count(word) * 5)
                
        # Low 가중치 (+2점)
        for word in rules['low']:
            if word in desc_clean:
                scores[category] += (desc_clean.count(word) * 2)
                
        # Negative 감점 (-5점)
        for word in rules['neg']:
            if word in desc_clean:
                scores[category] -= (desc_clean.count(word) * 5)
    
    # 최고 점수 추출
    max_score = max(scores.values())
    
    # [2단계] 점수가 0점 이하일 때, 1차 필터였던 tactics를 보고 기본값(Fallback) 지정!
    if max_score <= 0:
        if 'impact' in tactics_str:
            return 'Sabotage'      # 파괴(Impact) 전술인데 키워드가 없으면 기본 '사보타주'로 구제
        elif 'exfiltration' in tactics_str or 'collection' in tactics_str or 'credential-access' in tactics_str:
            return 'Espionage'     # 탈취/접근 전술인데 키워드가 없으면 기본 '에스피오나지'로 구제
        else:
            return 'General / Dual-Use'  # 진짜 대분류 전술과도 아무 연관 없으면 공통 범용 기술 처리
            
    # 최고 점수를 받은 카테고리 추출
    best_categories = [cat for cat, score in scores.items() if score == max_score]
    
    # 동점일 경우 듀얼(A & B) 방지를 위해 무조건 첫 번째 1개만 강제 선택
    return best_categories[0]

# 4. 데이터프레임 전체 행(row)에 하이브리드 함수 적용
df['Final_Category'] = df.apply(calculate_hybrid_category, axis=1)

# 5. 최종 결과 통계 확인
print("=== 하이브리드 분류 결과 통계 ===")
print(df['Final_Category'].value_counts())
print("-" * 50)

# 6. 상위 10개 행 눈으로 확인하기
print("=== 최종 분류 결과 샘플 (상위 10개) ===")
print(df[['technique_id', 'Final_Category']].head(10))
# df.head()

=== 하이브리드 분류 결과 통계 ===
Final_Category
Espionage             1872
General / Dual-Use    1728
Financial Theft        631
Data destruction       165
Sabotage                56
Denial of service       49
Doxing                  30
Defacement              15
Name: count, dtype: int64
--------------------------------------------------
=== 최종 분류 결과 샘플 (상위 10개) ===
  technique_id      Final_Category
0        T1003           Espionage
1        T1539           Espionage
2        T1490     Financial Theft
3        T1136  General / Dual-Use
4        T1204     Financial Theft
5        T1027           Espionage
6        T1059  General / Dual-Use
7        T1684  General / Dual-Use
8        T1571  General / Dual-Use
9        T1681           Espionage
